# Lecture 10 · RNN by hand · learn to count parentheses

**Goal** · build a vanilla RNN from scratch, train it on a toy task, see the vanishing-gradient problem live, and switch to LSTM to fix it.

Task · given a string of `(` and `)`, predict whether the parentheses are balanced. Requires counting — memory.

We'll:
1. Build a RNNCell from `nn.Linear` pieces.
2. Train it on short sequences (works).
3. Train on long sequences (breaks · vanishing gradient).
4. Swap in LSTM — works again.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

torch.manual_seed(0)
random.seed(0)

## 1 · Dataset · balanced parentheses

Generate strings of length L. Track the running count · positive if balanced at the end, negative if ever < 0.
Label · 1 if balanced, 0 if not.

In [ ]:
def gen_sequence(L):
    s = [random.choice(['(', ')']) for _ in range(L)]
    # label · balanced iff count(() == count()) and never negative running
    count = 0
    ok = True
    for c in s:
        count += 1 if c == '(' else -1
        if count < 0:
            ok = False
    balanced = 1 if ok and count == 0 else 0
    return s, balanced

def make_dataset(N, L):
    xs, ys = [], []
    for _ in range(N):
        s, y = gen_sequence(L)
        # one-hot encode '(' = [1, 0], ')' = [0, 1]
        x = torch.tensor([[1, 0] if c == '(' else [0, 1] for c in s], dtype=torch.float32)
        xs.append(x)
        ys.append(y)
    return torch.stack(xs), torch.tensor(ys, dtype=torch.float32)

xs, ys = make_dataset(1000, L=10)
print(f'train shape {xs.shape}  labels mean {ys.mean():.2f}')

## 2 · RNNCell by hand

In [ ]:
class VanillaRNN(nn.Module):
    def __init__(self, d_in=2, d_h=16):
        super().__init__()
        self.W = nn.Linear(d_in, d_h, bias=False)
        self.U = nn.Linear(d_h, d_h, bias=True)
        self.out = nn.Linear(d_h, 1)
        self.d_h = d_h

    def forward(self, x):
        # x: (batch, seq_len, d_in)
        B, L, _ = x.shape
        h = torch.zeros(B, self.d_h)
        for t in range(L):
            h = torch.tanh(self.W(x[:, t]) + self.U(h))
        return self.out(h).squeeze(-1)   # (batch,)

## 3 · Train on short sequences · RNN works

In [ ]:
def train_model(model, xs, ys, epochs=50, lr=1e-2):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(epochs):
        logits = model(xs)
        loss = F.binary_cross_entropy_with_logits(logits, ys)
        opt.zero_grad(); loss.backward(); opt.step()
        if ep % 10 == 0:
            pred = (torch.sigmoid(logits) > 0.5).float()
            acc = (pred == ys).float().mean()
            print(f'epoch {ep:3d}  loss {loss.item():.3f}  acc {acc.item():.2f}')
    return model

print('--- RNN on length 10 ---')
xs, ys = make_dataset(1000, L=10)
train_model(VanillaRNN(), xs, ys)

## 4 · Train on long sequences · RNN struggles

In [ ]:
print('--- RNN on length 50 ---')
xs, ys = make_dataset(1000, L=50)
train_model(VanillaRNN(), xs, ys)

**Observe** · accuracy stalls near 50% (= guessing). Vanishing gradients prevent the model from learning long-range counting.

## 5 · Swap to LSTM · problem solved

In [ ]:
class LSTMNet(nn.Module):
    def __init__(self, d_in=2, d_h=16):
        super().__init__()
        self.lstm = nn.LSTM(d_in, d_h, batch_first=True)
        self.out = nn.Linear(d_h, 1)
    def forward(self, x):
        h, _ = self.lstm(x)   # h: (batch, L, d_h)
        return self.out(h[:, -1]).squeeze(-1)

print('--- LSTM on length 50 ---')
xs, ys = make_dataset(1000, L=50)
train_model(LSTMNet(), xs, ys)

**Observe** · accuracy climbs past 90% within a few epochs. LSTM's gated cell state carries the count across 50 timesteps without losing gradient.

This is the practical meaning of the vanishing-gradient fix.